# Exploration Strategies & Model-Based RL

[← Back to lesson](https://ml-viz-ruby.vercel.app/courses/reinforcement-learning/05-exploration-and-model-based)

This notebook implements multi-armed bandit algorithms (ε-greedy, UCB, Thompson Sampling) on a k-arm Bernoulli bandit, simulates Dyna-Q on a simple grid world, and visualizes the sample efficiency gains from model-based planning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import defaultdict

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(42)

## 1. Multi-armed bandit: ε-greedy vs UCB vs Thompson Sampling

A 10-arm bandit where each arm has an unknown Bernoulli success probability. We compare three exploration strategies over 1000 timesteps.

In [ ]:
K = 10  # number of arms
T = 1000  # timesteps
N_RUNS = 200  # independent runs for averaging

# True arm success probabilities (fixed across runs for fair comparison)
TRUE_P = np.array([0.1, 0.25, 0.4, 0.55, 0.7, 0.6, 0.45, 0.3, 0.15, 0.5])
BEST_ARM = TRUE_P.argmax()


def run_epsilon_greedy(epsilon, T=T, seed=0):
    rng = np.random.RandomState(seed)
    Q = np.zeros(K)  # estimated value per arm
    N = np.zeros(K)  # visit count
    rewards = []
    for t in range(T):
        if rng.rand() < epsilon:
            arm = rng.randint(K)
        else:
            arm = Q.argmax()
        r = rng.rand() < TRUE_P[arm]
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]  # running mean
        rewards.append(r)
    return np.array(rewards)


def run_ucb(c=1.0, T=T, seed=0):
    rng = np.random.RandomState(seed)
    Q = np.zeros(K)
    N = np.zeros(K)
    rewards = []
    for t in range(T):
        if t < K:  # try each arm once
            arm = t
        else:
            bonus = c * np.sqrt(np.log(t + 1) / (N + 1e-9))
            arm = (Q + bonus).argmax()
        r = rng.rand() < TRUE_P[arm]
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)
    return np.array(rewards)


def run_thompson(T=T, seed=0):
    rng = np.random.RandomState(seed)
    alpha = np.ones(K)  # Beta(alpha, beta) posterior
    beta = np.ones(K)
    rewards = []
    for t in range(T):
        theta_sample = rng.beta(alpha, beta)
        arm = theta_sample.argmax()
        r = rng.rand() < TRUE_P[arm]
        if r:
            alpha[arm] += 1
        else:
            beta[arm] += 1
        rewards.append(r)
    return np.array(rewards)


# Average over N_RUNS
def avg_rewards(fn, **kwargs):
    return np.mean([fn(seed=s, **kwargs) for s in range(N_RUNS)], axis=0)

r_eps01 = avg_rewards(run_epsilon_greedy, epsilon=0.1)
r_eps30 = avg_rewards(run_epsilon_greedy, epsilon=0.3)
r_ucb   = avg_rewards(run_ucb, c=1.0)
r_ts    = avg_rewards(run_thompson)
optimal_r = TRUE_P[BEST_ARM]

# Cumulative average reward
ts = np.arange(1, T+1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ts, np.cumsum(r_eps01)/ts, label='ε-greedy (ε=0.1)', color='#6366f1', linewidth=2)
ax.plot(ts, np.cumsum(r_eps30)/ts, label='ε-greedy (ε=0.3)', color='#4b5563', linewidth=2)
ax.plot(ts, np.cumsum(r_ucb)/ts,   label='UCB (c=1.0)',       color='#f59e0b', linewidth=2)
ax.plot(ts, np.cumsum(r_ts)/ts,    label='Thompson Sampling',  color='#10b981', linewidth=2)
ax.axhline(optimal_r, color='white', linestyle='--', linewidth=1, alpha=0.5, label=f'Optimal (p={optimal_r})')
ax.set_xlabel('Timestep')
ax.set_ylabel('Average reward')
ax.set_title('Multi-Armed Bandit: Exploration Strategy Comparison', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal arm: {BEST_ARM} (p = {optimal_r})")
print(f"Final avg reward — ε=0.1: {r_eps01[-100:].mean():.3f}, UCB: {r_ucb[-100:].mean():.3f}, TS: {r_ts[-100:].mean():.3f}")

## 2. UCB regret — sub-linear growth

Regret = (optimal reward we could have gotten) − (reward we actually got). UCB achieves O(√T log T) regret.

In [ ]:
def cumulative_regret(rewards):
    return np.cumsum(optimal_r - rewards)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ts, cumulative_regret(r_eps01), label='ε-greedy (ε=0.1)', color='#6366f1', linewidth=2)
ax.plot(ts, cumulative_regret(r_ucb),   label='UCB (c=1.0)',       color='#f59e0b', linewidth=2)
ax.plot(ts, cumulative_regret(r_ts),    label='Thompson Sampling',  color='#10b981', linewidth=2)

# Reference curves
ax.plot(ts, 0.15 * np.sqrt(ts * np.log(ts+1)), 'w--', linewidth=1, alpha=0.5, label='O(√T log T)')
ax.plot(ts, 0.05 * ts, 'r--', linewidth=1, alpha=0.5, label='O(T) (linear)')

ax.set_xlabel('Timestep')
ax.set_ylabel('Cumulative regret')
ax.set_title('Cumulative Regret: UCB & Thompson Sampling are Sub-linear', color='#e2e8f0')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Dyna-Q: model-based planning for sample efficiency

A simple grid world: agent starts at (0,0), goal at (4,4). Compare Q-learning alone vs Dyna-Q with k planning steps.

In [ ]:
GRID = 5  # 5×5 grid
GOAL = (4, 4)
ACTIONS = [(0,1),(0,-1),(1,0),(-1,0)]  # right, left, down, up

def step_env(s, a_idx):
    r, c = s
    dr, dc = ACTIONS[a_idx]
    nr, nc = np.clip(r+dr, 0, GRID-1), np.clip(c+dc, 0, GRID-1)
    s_next = (nr, nc)
    reward = 1.0 if s_next == GOAL else 0.0
    done = (s_next == GOAL)
    return s_next, reward, done

def run_dyna_q(k_plan=0, n_episodes=200, alpha=0.5, gamma=0.9, eps=0.1, seed=0):
    rng = np.random.RandomState(seed)
    Q = np.zeros((GRID, GRID, 4))
    model = {}  # (s, a) → (r, s')
    visited = []  # list of (s, a) pairs
    episode_lengths = []

    for ep in range(n_episodes):
        s = (0, 0)
        steps = 0
        while True:
            # ε-greedy action
            if rng.rand() < eps:
                a = rng.randint(4)
            else:
                a = Q[s].argmax()

            s_next, r, done = step_env(s, a)

            # Q-learning update (real step)
            Q[s][a] += alpha * (r + gamma * Q[s_next].max() - Q[s][a])

            # Update model
            model[(s, a)] = (r, s_next)
            if (s, a) not in visited:
                visited.append((s, a))

            # Dyna planning steps
            for _ in range(k_plan):
                if not visited:
                    break
                idx = rng.randint(len(visited))
                ps, pa = visited[idx]
                pr, ps_next = model[(ps, pa)]
                Q[ps][pa] += alpha * (pr + gamma * Q[ps_next].max() - Q[ps][pa])

            s = s_next
            steps += 1
            if done or steps > 500:
                break

        episode_lengths.append(steps)

    return episode_lengths

# Compare different k values
k_values = [0, 5, 25, 50]
colors = ['#4b5563', '#6366f1', '#f59e0b', '#10b981']

fig, ax = plt.subplots(figsize=(10, 5))
for k, color in zip(k_values, colors):
    lengths = np.mean([run_dyna_q(k_plan=k, seed=s) for s in range(20)], axis=0)
    # Smooth
    smooth = np.convolve(lengths, np.ones(10)/10, mode='valid')
    label = f'k={k} ({"Q-learning" if k==0 else f"Dyna-Q, {k} planning steps"})'
    ax.plot(smooth, color=color, linewidth=2, label=label)

ax.set_xlabel('Episode')
ax.set_ylabel('Steps to reach goal (lower = faster)')
ax.set_title('Dyna-Q: Planning Steps Improve Sample Efficiency', color='#e2e8f0')
ax.legend()
ax.set_ylim(0, 200)
plt.tight_layout()
plt.show()

print("Key insight: more planning steps (k) → faster convergence with the same number of real steps.")

## ✏️ Your turn

**Exercise 1 — Optimistic initialization.** ε-greedy with ε=0.0 (pure greedy) normally fails — it exploits the first arm tried forever. But initialize Q to a large positive value (e.g., Q = 5.0 for all arms) and it explores naturally. Implement this and compare with ε-greedy (ε=0.1) over 1000 timesteps.

In [ ]:
def run_optimistic(Q_init=5.0, T=T, seed=0):
    """
    Pure greedy (ε=0) with optimistic initialization.
    TODO(you): implement and compare against ε-greedy
    """
    rng = np.random.RandomState(seed)
    Q = np.full(K, Q_init)  # all arms start optimistic
    N = np.zeros(K)
    rewards = []
    # TODO: at each step, take greedy action (no exploration noise needed)
    return np.array(rewards)

In [ ]:
# Assert cell
def run_optimistic_ref(Q_init=5.0, T=T, seed=0):
    rng = np.random.RandomState(seed)
    Q = np.full(K, float(Q_init))
    N = np.zeros(K)
    rewards = []
    for t in range(T):
        arm = Q.argmax()
        r = rng.rand() < TRUE_P[arm]
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)
    return np.array(rewards)

r_opt = avg_rewards(run_optimistic_ref, Q_init=5.0)
print(f"Optimistic init (Q₀=5) final avg reward: {r_opt[-100:].mean():.3f}")
print(f"ε-greedy (ε=0.1) final avg reward:       {r_eps01[-100:].mean():.3f}")
assert r_opt[-100:].mean() > 0.5, "Optimistic init should converge near the optimal arm"

<details><summary>Solution</summary>

```python
def run_optimistic(Q_init=5.0, T=T, seed=0):
    rng = np.random.RandomState(seed)
    Q = np.full(K, float(Q_init))
    N = np.zeros(K)
    rewards = []
    for t in range(T):
        arm = Q.argmax()           # pure greedy — no ε needed!
        r = rng.rand() < TRUE_P[arm]
        N[arm] += 1
        Q[arm] += (r - Q[arm]) / N[arm]
        rewards.append(r)
    return np.array(rewards)
```

Optimistic initialization works because any arm tried will get its Q-value dragged downward (since true rewards are ≪ 5). Untried arms still look optimistic, so the agent naturally tries each arm at least once. After initial exploration, it converges to exploiting the best arm. The effect fades over time as Q-values converge to their true values.

</details>